<a href="https://colab.research.google.com/github/jaiswalvik/MathematicalFoundationsOfGenerativeAI/blob/master/Assignment2_VanillaGAN_DCGAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 2 — Vanilla GAN, DCGAN, and Conditional DCGAN

This notebook includes the required unconditional Vanilla GAN and DCGAN baselines, plus an optional class-conditional DCGAN using a projection discriminator. The dataset split and image transforms are aligned with the uploaded Assignment 1 notebook. It includes training curves, generated grids, latent interpolation, held-out evaluation (FID when available, otherwise a clearly labelled proxy), best-checkpoint saving, and controlled sensitivity experiment code.

**Smoke-test configuration:** `EPOCHS = 1`. Once the notebook runs end-to-end, change it to `50`. Set `DATA_DIR` / `ZIP_PATH` to the dataset used in Assignment 1. Preprocessing and split logic are aligned with Assignment 1: RandomResizedCrop + RandomHorizontalFlip for training; Resize(256) + CenterCrop(224) for validation/test; ImageNet mean/std; stratified 80/20 split with seed 42. No dataset is embedded in this notebook.


## 1. Install / import dependencies

In [ ]:
%pip -q install scikit-learn matplotlib pandas pillow

In [ ]:
import os, json, time, random, copy, math, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, utils
from sklearn.model_selection import train_test_split

print("PyTorch:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cpu":
    print("Warning: GAN training, especially at 224x224, may be very slow on CPU.")


## 2. Mount Google Drive and configure paths

Expected structure: `dataset1/train/<class folders>` and `dataset1/test/<class folders>`. The notebook creates a seeded stratified validation split from the training folder and holds the test folder out. If Assignment 1 used a different split, reproduce it here.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Point this to the same dataset folder used in Assignment 1.
DATA_DIR = Path("/content/dataset1")
ZIP_PATH = Path("/content/drive/MyDrive/dataset1.zip")
if not (DATA_DIR / "train").is_dir() and ZIP_PATH.is_file():
    import zipfile
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall("/content")

OUTPUT_DIR = Path("/content/drive/MyDrive/assignment2_gan_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 1  # Smoke test; change to 50 after confirming all cells work.
LATENT_DIM = 100
NUM_WORKERS = 2
LEARNING_RATE = 2e-4
BETA1, BETA2 = 0.5, 0.999
VAL_FRACTION = 0.20
SAMPLE_EVERY = 1
EARLY_STOPPING_PATIENCE = 5
RUN_SENSITIVITY_EXPERIMENTS = False
RUN_CONDITIONAL_DCGAN = True

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
assert TRAIN_DIR.is_dir(), f"Training folder not found: {TRAIN_DIR}. Set DATA_DIR or provide ZIP_PATH."
assert TEST_DIR.is_dir(), f"Test folder not found: {TEST_DIR}"
print("Dataset:", DATA_DIR)
print("Output:", OUTPUT_DIR)


## 3. Dataset, transforms, and DataLoaders

**Matched to Assignment 1:** use the same `ImageFolder` ordering, stratified 80/20 split (`random_state=42`), training augmentation, deterministic evaluation transforms, ImageNet mean/std, batch size (16), and worker count (2). Separate dataset objects are necessary because the training subset uses random augmentation while validation uses deterministic preprocessing.

The generators emit RGB in `[-1, 1]`; `generator_to_discriminator_scale()` converts generated images to ImageNet-normalized space before the discriminator sees them. Real images use the Assignment 1 ImageNet normalization. This keeps real and fake discriminator inputs on the same numeric scale.


In [ ]:
# Match Assignment 1 preprocessing exactly: ImageNet normalization, random resized
# crop + horizontal flip for training, Resize(256) + CenterCrop(224) for val/test.
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Separate train/eval dataset instances share the same file ordering. This matches
# Assignment 1 and lets the same split indices use augmentation only for training.
train_dataset_aug = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
train_dataset_eval = datasets.ImageFolder(TRAIN_DIR, transform=eval_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_transform)

class_names = train_dataset_aug.classes
num_classes = len(class_names)
class_to_idx = train_dataset_aug.class_to_idx
assert train_dataset_eval.class_to_idx == class_to_idx, "Train dataset mappings differ."
assert test_dataset.class_to_idx == class_to_idx, "Train/test class folder mappings differ."
assert train_dataset_aug.samples == train_dataset_eval.samples, "Train dataset file ordering differs across transforms."
print("Classes:", num_classes, class_names)
print("Class-to-index mapping:", class_to_idx)
if num_classes != 10:
    print(f"Note: found {num_classes} classes; the assignment describes a 10-class task.")
print("Train images:", len(train_dataset_aug), "| Test images:", len(test_dataset))

indices = np.arange(len(train_dataset_aug))
targets = np.array(train_dataset_aug.targets)
train_idx, val_idx = train_test_split(
    indices, test_size=VAL_FRACTION, random_state=SEED, stratify=targets
)
train_subset = Subset(train_dataset_aug, train_idx)
val_subset = Subset(train_dataset_eval, val_idx)
loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)
# Assignment 1 keeps the final partial batch, so drop_last=False here too.
train_loader = DataLoader(train_subset, shuffle=True, drop_last=False, **loader_kwargs)
val_loader = DataLoader(val_subset, shuffle=False, drop_last=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, drop_last=False, **loader_kwargs)
print(f"Train split: {len(train_subset)} | Validation split: {len(val_subset)} | Test: {len(test_dataset)}")
print(f"Train batches: {len(train_loader)} | Validation batches: {len(val_loader)} | Test batches: {len(test_loader)}")


## 4. Inspect a real-image batch

In [ ]:
real_batch, label_batch = next(iter(train_loader))
mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)
real_display = (real_batch.cpu() * std + mean).clamp(0, 1)
grid = utils.make_grid(real_display[:16], nrow=4)
plt.figure(figsize=(7, 7))
plt.imshow(grid.permute(1, 2, 0))
plt.axis("off")
plt.title("Real training images")
plt.show()


## 5. Define the Vanilla GAN

- **Generator**: maps a random latent vector `z ~ N(0, I)` to a flattened image vector, then reshapes it into an image.
- **Discriminator**: maps an image to one logit; positive logits indicate more confidence that an image is real.
- **Loss**: `BCEWithLogitsLoss`, numerically stable binary cross entropy.

At 224×224, fully connected GAN layers are large. If memory is insufficient, reduce `IMAGE_SIZE` to 64 and rerun all data/model cells, then clearly note the resolution change in your report.

In [ ]:
class VanillaGenerator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, image_size=IMAGE_SIZE, channels=3):
        super().__init__()
        self.image_size = image_size
        self.channels = channels
        out_dim = channels * image_size * image_size
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(True),
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(True),
            nn.Linear(1024, out_dim),
            nn.Tanh(),
        )

    def forward(self, z):
        x = self.net(z)
        return x.view(z.size(0), self.channels, self.image_size, self.image_size)


class VanillaDiscriminator(nn.Module):
    def __init__(self, image_size=IMAGE_SIZE, channels=3):
        super().__init__()
        in_dim = channels * image_size * image_size
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_dim, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, 1),  # logits; no sigmoid with BCEWithLogitsLoss
        )

    def forward(self, x):
        return self.net(x)


vanilla_G = VanillaGenerator().to(device)
vanilla_D = VanillaDiscriminator().to(device)
print("Vanilla Generator parameters:", sum(p.numel() for p in vanilla_G.parameters()))
print("Vanilla Discriminator parameters:", sum(p.numel() for p in vanilla_D.parameters()))

## 6. Define DCGAN

The DCGAN uses strided convolutions in the discriminator and transposed convolutions in the generator, avoiding pooling layers. Batch normalization and ReLU are used in the generator; LeakyReLU is used in the discriminator. The implementation supports common power-of-two image sizes (64, 128) and 224 by using a 7×7 starting feature map followed by ×2 upsampling stages. The final layer is resized to the configured spatial resolution if needed.

In [ ]:
class DCGANGenerator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, image_size=IMAGE_SIZE, channels=3, base=64):
        super().__init__()
        self.image_size = image_size
        # Start at 7x7 for 224px images; repeatedly double spatial dimensions
        # until reaching or exceeding the target, then resize only if necessary.
        start_size = 7
        stages = max(1, int(math.ceil(math.log2(image_size / start_size))))
        self.start_size = start_size
        first_channels = base * 8
        self.fc = nn.Sequential(
            nn.Linear(latent_dim, first_channels * start_size * start_size),
            nn.BatchNorm1d(first_channels * start_size * start_size),
            nn.ReLU(True),
        )
        layers = []
        in_ch = first_channels
        # Decrease channels as spatial dimensions increase.
        channel_schedule = [base * 4, base * 2, base, max(base // 2, 8), max(base // 4, 8)]
        for stage in range(stages):
            out_ch = channel_schedule[min(stage, len(channel_schedule) - 1)]
            layers += [
                nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(True),
            ]
            in_ch = out_ch
        layers += [nn.Conv2d(in_ch, channels, kernel_size=3, stride=1, padding=1), nn.Tanh()]
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        x = self.fc(z).view(z.size(0), -1, self.start_size, self.start_size)
        x = self.net(x)
        if x.shape[-1] != self.image_size or x.shape[-2] != self.image_size:
            x = torch.nn.functional.interpolate(
                x, size=(self.image_size, self.image_size), mode="bilinear", align_corners=False
            )
        return x

In [ ]:
class DCGANDiscriminator(nn.Module):
    def __init__(self, image_size=IMAGE_SIZE, channels=3, base=64):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(channels, base, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(base, base * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base * 2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(base * 2, base * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base * 4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(base * 4, base * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base * 8),
            nn.LeakyReLU(0.2, inplace=True),

            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.classifier = nn.Linear(base * 8, 1)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)

## 7. Losses, optimizers, and image-grid utilities

We use the non-saturating generator loss: the generator tries to make the discriminator classify fake images as real. The discriminator learns to classify real images as real and generated images as fake. One discriminator update and one generator update are performed per batch.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
IMAGENET_MEAN_T = torch.tensor(IMAGENET_MEAN, device=device).view(1, 3, 1, 1)
IMAGENET_STD_T = torch.tensor(IMAGENET_STD, device=device).view(1, 3, 1, 1)

def generator_to_discriminator_scale(fake_tanh):
    # G emits RGB in [-1,1]. Convert to [0,1], then apply Assignment 1 normalization.
    rgb01 = ((fake_tanh + 1.0) / 2.0).clamp(0, 1)
    return (rgb01 - IMAGENET_MEAN_T) / IMAGENET_STD_T

def real_to_display_scale(real_normalized):
    return (real_normalized * IMAGENET_STD_T + IMAGENET_MEAN_T).clamp(0, 1)

def init_dcgan_weights(module):
    if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(module.weight.data, 0.0, 0.02)
        if module.bias is not None:
            nn.init.constant_(module.bias.data, 0)

def make_optimizers(G, D, lr=LEARNING_RATE):
    opt_G = optim.Adam(G.parameters(), lr=lr, betas=(BETA1, BETA2))
    opt_D = optim.Adam(D.parameters(), lr=lr, betas=(BETA1, BETA2))
    return opt_G, opt_D

fixed_noise = torch.randn(64, LATENT_DIM, device=device)

def save_sample_grid(G, model_name, epoch, noise=None):
    if noise is None:
        noise = torch.randn(64, LATENT_DIM, device=device)
    G.eval()
    with torch.no_grad():
        fake = G(noise).detach().cpu()
    grid = utils.make_grid(fake, nrow=8, normalize=True, value_range=(-1, 1))
    path = OUTPUT_DIR / f"{model_name}_samples_epoch_{epoch:03d}.png"
    plt.figure(figsize=(10, 10))
    plt.imshow(grid.permute(1, 2, 0).clamp(0, 1))
    plt.axis("off")
    plt.title(f"{model_name} generated samples — epoch {epoch}")
    plt.savefig(path, bbox_inches="tight")
    plt.show()
    plt.close()
    G.train()
    return str(path)


## 8. Train one GAN

For validation, this notebook monitors validation discriminator/generator losses as a diagnostic proxy. GAN validation loss is not a reliable image-quality metric by itself, so the final report should include visual sample grids and FID/KID or an explicitly labelled proxy metric.

In [ ]:
@torch.no_grad()
def validate_gan(G, D):
    # Validation GAN losses are diagnostic proxies, not reliable image-quality scores.
    G.eval(); D.eval()
    d_total = g_total = 0.0
    count = 0
    for real, _labels in val_loader:
        real = real.to(device)
        bs = real.size(0)
        fake = generator_to_discriminator_scale(G(torch.randn(bs, LATENT_DIM, device=device)))
        real_loss = criterion(D(real), torch.ones(bs, 1, device=device))
        fake_loss = criterion(D(fake), torch.zeros(bs, 1, device=device))
        d_total += (real_loss + fake_loss).item()
        g_total += criterion(D(fake), torch.ones(bs, 1, device=device)).item()
        count += 1
    G.train(); D.train()
    return d_total / max(1, count), g_total / max(1, count)


def plot_gan_history(history, model_name):
    plt.figure(figsize=(9, 4))
    for key, label, style in [
        ("d_loss", "Train D loss", "-"), ("g_loss", "Train G loss", "-"),
        ("val_d_loss", "Validation D loss", "--"), ("val_g_loss", "Validation G loss", "--")
    ]:
        plt.plot(history[key], style, label=label)
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title(f"{model_name} training curves")
    plt.legend(); plt.grid(True, alpha=0.25)
    path = OUTPUT_DIR / f"{model_name}_loss_curves.png"
    plt.savefig(path, bbox_inches="tight"); plt.show(); plt.close()
    print("Saved:", path)


def train_gan(model_name, G, D, epochs=EPOCHS, label_smoothing=True,
              input_noise_std=0.0, early_stopping_patience=EARLY_STOPPING_PATIENCE, learning_rate=LEARNING_RATE):
    G.apply(init_dcgan_weights); D.apply(init_dcgan_weights)
    G.train(); D.train()
    opt_G, opt_D = make_optimizers(G, D, lr=learning_rate)
    history = {k: [] for k in ["d_loss", "g_loss", "val_d_loss", "val_g_loss"]}
    sample_paths = []
    best_val_g, best_epoch, stale = float("inf"), 0, 0

    for epoch in range(1, epochs + 1):
        d_running = g_running = 0.0
        n_batches = 0
        for real, _labels in train_loader:
            real = real.to(device, non_blocking=True)
            bs = real.size(0)
            real_targets = torch.full((bs, 1), 0.9 if label_smoothing else 1.0, device=device)
            fake_targets = torch.zeros(bs, 1, device=device)
            real_in = real + input_noise_std * torch.randn_like(real) if input_noise_std else real

            opt_D.zero_grad(set_to_none=True)
            z = torch.randn(bs, LATENT_DIM, device=device)
            with torch.no_grad():
                fake = generator_to_discriminator_scale(G(z))
            fake_in = fake + input_noise_std * torch.randn_like(fake) if input_noise_std else fake
            loss_D = criterion(D(real_in), real_targets) + criterion(D(fake_in.detach()), fake_targets)
            loss_D.backward(); opt_D.step()

            opt_G.zero_grad(set_to_none=True)
            z = torch.randn(bs, LATENT_DIM, device=device)
            generated = generator_to_discriminator_scale(G(z))
            loss_G = criterion(D(generated), torch.ones(bs, 1, device=device))
            loss_G.backward(); opt_G.step()
            d_running += loss_D.item(); g_running += loss_G.item(); n_batches += 1

        train_d, train_g = d_running/max(1,n_batches), g_running/max(1,n_batches)
        val_d, val_g = validate_gan(G, D)
        for key, value in [("d_loss",train_d),("g_loss",train_g),("val_d_loss",val_d),("val_g_loss",val_g)]:
            history[key].append(float(value))
        print(f"{model_name} | epoch {epoch:03d}/{epochs} | D={train_d:.4f} G={train_g:.4f} | val D={val_d:.4f} G={val_g:.4f}")

        if epoch == 1 or epoch % SAMPLE_EVERY == 0 or epoch == epochs:
            sample_paths.append(save_sample_grid(G, model_name, epoch))
        torch.save(G.state_dict(), OUTPUT_DIR / f"{model_name}_generator_latest.pth")
        torch.save(D.state_dict(), OUTPUT_DIR / f"{model_name}_discriminator_latest.pth")

        # Weak validation proxy only; final selection should consider FID/KID and sample quality.
        if val_g < best_val_g:
            best_val_g, best_epoch, stale = val_g, epoch, 0
            torch.save(G.state_dict(), OUTPUT_DIR / f"{model_name}_generator_best.pth")
            torch.save(D.state_dict(), OUTPUT_DIR / f"{model_name}_discriminator_best.pth")
        else:
            stale += 1
        if early_stopping_patience and stale >= early_stopping_patience:
            print(f"Early stopping at epoch {epoch}; best validation G-loss proxy at epoch {best_epoch}.")
            break

    with open(OUTPUT_DIR / f"{model_name}_history.json", "w") as f:
        json.dump(history, f, indent=2)
    plot_gan_history(history, model_name)
    print(f"Best checkpoint epoch (validation G-loss proxy): {best_epoch}")
    return history, sample_paths


## 9. Train Vanilla GAN

Run this first. If you encounter CUDA out-of-memory at 224×224, reduce `IMAGE_SIZE` and `BATCH_SIZE`, then rerun the notebook from the dataset/model cells.

In [ ]:
vanilla_G = VanillaGenerator().to(device)
vanilla_D = VanillaDiscriminator().to(device)
vanilla_history, vanilla_samples = train_gan(
    "vanilla_gan", vanilla_G, vanilla_D,
    epochs=EPOCHS,
    label_smoothing=True,
    input_noise_std=0.0,
)

## 10. Train DCGAN

In [ ]:
dcgan_G = DCGANGenerator().to(device)
dcgan_D = DCGANDiscriminator().to(device)
dcgan_history, dcgan_samples = train_gan(
    "dcgan", dcgan_G, dcgan_D,
    epochs=EPOCHS,
    label_smoothing=True,
    input_noise_std=0.0,
)

## 11. Optional extension: class-conditional DCGAN

The conditional Generator combines noise `z` with a learned class embedding. The projection Discriminator scores both image realism and compatibility with the requested class. `ImageFolder` supplies the labels. This optional model does not replace either required baseline.


In [ ]:
class ConditionalDCGANGenerator(nn.Module):
    def __init__(self, num_classes, latent_dim=LATENT_DIM, image_size=IMAGE_SIZE,
                 channels=3, base=64, label_dim=50):
        super().__init__()
        self.image_size, self.channels = image_size, channels
        self.label_embedding = nn.Embedding(num_classes, label_dim)
        self.start_size = 7
        stages = max(1, int(math.ceil(math.log2(image_size / self.start_size))))
        first_channels = base * 8
        self.fc = nn.Sequential(
            nn.Linear(latent_dim + label_dim, first_channels * self.start_size * self.start_size),
            nn.BatchNorm1d(first_channels * self.start_size * self.start_size),
            nn.ReLU(True),
        )
        layers, in_ch = [], first_channels
        schedule = [base*4, base*2, base, max(base//2,8), max(base//4,8)]
        for stage in range(stages):
            out_ch = schedule[min(stage, len(schedule)-1)]
            layers += [nn.ConvTranspose2d(in_ch,out_ch,4,2,1,bias=False),
                       nn.BatchNorm2d(out_ch), nn.ReLU(True)]
            in_ch = out_ch
        layers += [nn.Conv2d(in_ch,channels,3,1,1), nn.Tanh()]
        self.net = nn.Sequential(*layers)

    def forward(self, z, labels):
        y = self.label_embedding(labels)
        x = self.fc(torch.cat([z,y],dim=1))
        x = x.view(z.size(0),-1,self.start_size,self.start_size)
        x = self.net(x)
        if x.shape[-2:] != (self.image_size,self.image_size):
            x = torch.nn.functional.interpolate(x,size=(self.image_size,self.image_size),
                                                mode="bilinear",align_corners=False)
        return x


class ProjectionDiscriminator(nn.Module):
    def __init__(self, num_classes, channels=3, base=64):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(channels,base,4,2,1), nn.LeakyReLU(0.2,inplace=True),
            nn.Conv2d(base,base*2,4,2,1,bias=False), nn.BatchNorm2d(base*2), nn.LeakyReLU(0.2,inplace=True),
            nn.Conv2d(base*2,base*4,4,2,1,bias=False), nn.BatchNorm2d(base*4), nn.LeakyReLU(0.2,inplace=True),
            nn.Conv2d(base*4,base*8,4,2,1,bias=False), nn.BatchNorm2d(base*8), nn.LeakyReLU(0.2,inplace=True),
            nn.AdaptiveAvgPool2d((1,1)))
        feature_dim = base*8
        self.unconditional = nn.Linear(feature_dim,1)
        self.class_embedding = nn.Embedding(num_classes,feature_dim)

    def forward(self, images, labels):
        features = self.features(images).flatten(1)
        score = self.unconditional(features).squeeze(1)
        projection = (self.class_embedding(labels)*features).sum(dim=1)/math.sqrt(features.size(1))
        return (score+projection).unsqueeze(1)


def train_conditional_dcgan(G,D,epochs=EPOCHS,label_smoothing=True,
                            early_stopping_patience=EARLY_STOPPING_PATIENCE):
    G.apply(init_dcgan_weights); D.apply(init_dcgan_weights)
    opt_G,opt_D = make_optimizers(G,D)
    history = {k:[] for k in ["d_loss","g_loss","val_d_loss","val_g_loss"]}
    best_val,best_epoch,stale = float("inf"),0,0
    for epoch in range(1,epochs+1):
        G.train(); D.train()
        dsum=gsum=0.0; batches=0
        for real,labels in train_loader:
            real,labels=real.to(device),labels.to(device); bs=real.size(0)
            rt=torch.full((bs,1),0.9 if label_smoothing else 1.0,device=device)
            ft=torch.zeros(bs,1,device=device)
            opt_D.zero_grad(set_to_none=True)
            with torch.no_grad(): fake=generator_to_discriminator_scale(G(torch.randn(bs,LATENT_DIM,device=device),labels))
            lossD=criterion(D(real,labels),rt)+criterion(D(fake.detach(),labels),ft)
            lossD.backward(); opt_D.step()
            opt_G.zero_grad(set_to_none=True)
            generated=generator_to_discriminator_scale(G(torch.randn(bs,LATENT_DIM,device=device),labels))
            lossG=criterion(D(generated,labels),torch.ones(bs,1,device=device))
            lossG.backward(); opt_G.step()
            dsum+=lossD.item(); gsum+=lossG.item(); batches+=1

        G.eval(); D.eval(); vd=vg=0.0; vb=0
        with torch.no_grad():
            for real,labels in val_loader:
                real,labels=real.to(device),labels.to(device); bs=real.size(0)
                fake=generator_to_discriminator_scale(G(torch.randn(bs,LATENT_DIM,device=device),labels))
                vd+=(criterion(D(real,labels),torch.ones(bs,1,device=device))+
                     criterion(D(fake,labels),torch.zeros(bs,1,device=device))).item()
                vg+=criterion(D(fake,labels),torch.ones(bs,1,device=device)).item(); vb+=1
        vals=[dsum/max(1,batches),gsum/max(1,batches),vd/max(1,vb),vg/max(1,vb)]
        for key,val in zip(history,vals): history[key].append(float(val))
        print(f"Conditional DCGAN | epoch {epoch:03d}/{epochs} | D={vals[0]:.4f} G={vals[1]:.4f} | val D={vals[2]:.4f} G={vals[3]:.4f}")
        torch.save(G.state_dict(),OUTPUT_DIR/"conditional_dcgan_generator_latest.pth")
        torch.save(D.state_dict(),OUTPUT_DIR/"conditional_dcgan_discriminator_latest.pth")
        if vals[3]<best_val:
            best_val,best_epoch,stale=vals[3],epoch,0
            torch.save(G.state_dict(),OUTPUT_DIR/"conditional_dcgan_generator_best.pth")
            torch.save(D.state_dict(),OUTPUT_DIR/"conditional_dcgan_discriminator_best.pth")
        else: stale+=1
        if early_stopping_patience and stale>=early_stopping_patience:
            print(f"Early stopping; best validation G-loss proxy at epoch {best_epoch}."); break
    with open(OUTPUT_DIR/"conditional_dcgan_history.json","w") as f: json.dump(history,f,indent=2)
    plot_gan_history(history,"conditional_dcgan")
    return history


@torch.no_grad()
def save_conditional_class_grids(G,samples_per_class=8):
    G.eval()
    for class_idx,class_name in enumerate(class_names):
        labels=torch.full((samples_per_class,),class_idx,dtype=torch.long,device=device)
        fake=G(torch.randn(samples_per_class,LATENT_DIM,device=device),labels).cpu()
        grid=utils.make_grid(fake,nrow=4,normalize=True,value_range=(-1,1))
        safe_name=re.sub(r"[^A-Za-z0-9_-]+","_",class_name)
        path=OUTPUT_DIR/f"conditional_dcgan_class_{class_idx}_{safe_name}.png"
        plt.figure(figsize=(7,7)); plt.imshow(grid.permute(1,2,0).clamp(0,1))
        plt.axis("off"); plt.title(f"Conditional DCGAN — class: {class_name}")
        plt.savefig(path,bbox_inches="tight"); plt.show(); plt.close()
        print("Saved:",path)
    G.train()


if RUN_CONDITIONAL_DCGAN:
    conditional_G=ConditionalDCGANGenerator(num_classes=num_classes).to(device)
    conditional_D=ProjectionDiscriminator(num_classes=num_classes).to(device)
    conditional_history=train_conditional_dcgan(conditional_G,conditional_D,epochs=EPOCHS)
    save_conditional_class_grids(conditional_G)
else:
    print("Conditional DCGAN skipped.")


## 12. Latent-space interpolation

Interpolate between two latent vectors for each unconditional generator. For the conditional model, labels are held fixed in each class-wise grid.


In [ ]:
@torch.no_grad()
def save_latent_interpolation(G, model_name, steps=8):
    G.eval()
    z1 = torch.randn(1, LATENT_DIM, device=device)
    z2 = torch.randn(1, LATENT_DIM, device=device)
    alphas = torch.linspace(0, 1, steps, device=device)
    zs = torch.cat([(1-a)*z1 + a*z2 for a in alphas], dim=0)
    imgs = G(zs).cpu()
    grid = utils.make_grid(imgs, nrow=steps, normalize=True, value_range=(-1, 1))
    plt.figure(figsize=(steps * 1.6, 2.5))
    plt.imshow(grid.permute(1, 2, 0).clamp(0, 1))
    plt.axis("off"); plt.title(f"{model_name}: latent interpolation")
    path = OUTPUT_DIR / f"{model_name}_latent_interpolation.png"
    plt.savefig(path, bbox_inches="tight"); plt.show(); plt.close()
    G.train()
    print("Saved:", path)

save_latent_interpolation(vanilla_G, "vanilla_gan")
save_latent_interpolation(dcgan_G, "dcgan")

## 13. Quantitative evaluation: FID when available, otherwise a proxy

FID normally compares feature distributions of real and generated images; lower is better. It is not a pixelwise accuracy metric. Inception Score can be less meaningful for a small or specialized dataset.

This cell attempts FID with `torchmetrics`. If the dependency or pretrained feature extractor is unavailable, it reports a simple feature-space nearest-neighbor proxy using resized flattened images. **That fallback is not FID and must be labelled as a proxy in the report.**


In [ ]:
@torch.no_grad()
def generate_images(G, n):
    G.eval()
    chunks = []
    remaining = n
    while remaining > 0:
        bs = min(BATCH_SIZE, remaining)
        fake = G(torch.randn(bs, LATENT_DIM, device=device)).cpu()
        chunks.append(((fake + 1) / 2).clamp(0, 1))
        remaining -= bs
    G.train()
    return torch.cat(chunks, dim=0)


@torch.no_grad()
def collect_real_images(loader, n):
    chunks, remaining = [], n
    mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)
    for real, _ in loader:
        bs = min(real.size(0), remaining)
        chunks.append((real[:bs].cpu() * std + mean).clamp(0, 1))
        remaining -= bs
        if remaining <= 0:
            break
    return torch.cat(chunks, dim=0)


def proxy_nearest_neighbor_distance(real_imgs, fake_imgs, side=16):
    # Rough pixel-space proxy, not FID/KID.
    real = torch.nn.functional.interpolate(real_imgs, size=(side, side), mode="bilinear", align_corners=False)
    fake = torch.nn.functional.interpolate(fake_imgs, size=(side, side), mode="bilinear", align_corners=False)
    real = torch.nn.functional.normalize(real.flatten(1), dim=1)
    fake = torch.nn.functional.normalize(fake.flatten(1), dim=1)
    return torch.cdist(fake, real, p=2).min(dim=1).values.mean().item()


def evaluate_generator(G, model_name, n_samples=256):
    real = collect_real_images(test_loader, min(n_samples, len(test_dataset)))
    fake = generate_images(G, len(real))
    results = {"model": model_name, "n_real": int(len(real)), "n_fake": int(len(fake))}
    try:
        from torchmetrics.image.fid import FrechetInceptionDistance
        fid_metric = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
        fid_metric.update(real.to(device), real=True)
        fid_metric.update(fake.to(device), real=False)
        results["FID"] = float(fid_metric.compute().item())
        results["metric_note"] = "FID; lower is better"
    except Exception as exc:
        results["proxy_nearest_neighbor_distance"] = proxy_nearest_neighbor_distance(real, fake)
        results["metric_note"] = "Pixel-space nearest-neighbor proxy, NOT FID/KID"
        results["fid_error"] = str(exc)[:300]
    grid = utils.make_grid(fake[:64], nrow=8)
    plt.figure(figsize=(9, 9))
    plt.imshow(grid.permute(1, 2, 0).clamp(0, 1)); plt.axis("off")
    plt.title(f"{model_name}: generated evaluation samples")
    plt.savefig(OUTPUT_DIR / f"{model_name}_evaluation_grid.png", bbox_inches="tight")
    plt.show(); plt.close()
    return results

vanilla_metrics = evaluate_generator(vanilla_G, "vanilla_gan")
dcgan_metrics = evaluate_generator(dcgan_G, "dcgan")
metrics = [vanilla_metrics, dcgan_metrics]
with open(OUTPUT_DIR / "quantitative_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
pd.DataFrame(metrics)


## 14. Controlled sensitivity experiments

A real experiment runner is provided below. It is disabled by default so the one-epoch smoke test does not trigger additional runs. Enable `RUN_SENSITIVITY_EXPERIMENTS=True` after the baseline works; keep all but one setting fixed for each comparison.


In [ ]:
experiment_plan = pd.DataFrame([
    {"experiment": "Baseline", "latent_dim": LATENT_DIM, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE, "stabilization": "one-sided label smoothing"},
    {"experiment": "Latent dim 64", "latent_dim": 64, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE, "stabilization": "same as baseline"},
    {"experiment": "Latent dim 128", "latent_dim": 128, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE, "stabilization": "same as baseline"},
    {"experiment": "Batch size 8", "latent_dim": LATENT_DIM, "batch_size": 8, "learning_rate": LEARNING_RATE, "stabilization": "same as baseline"},
    {"experiment": "Lower learning rate", "latent_dim": LATENT_DIM, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE/2, "stabilization": "same as baseline"},
    {"experiment": "No label smoothing", "latent_dim": LATENT_DIM, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE, "stabilization": "disabled"},
])
display(experiment_plan)

def run_sensitivity_experiments(epochs=EPOCHS):
    global LATENT_DIM, LEARNING_RATE, BATCH_SIZE, fixed_noise, train_loader, val_loader, test_loader
    configs = [
        {"name":"latent64","latent_dim":64,"lr":LEARNING_RATE,"smooth":True},
        {"name":"latent128","latent_dim":128,"lr":LEARNING_RATE,"smooth":True},
        {"name":"batch8","latent_dim":LATENT_DIM,"lr":LEARNING_RATE,"smooth":True,"batch_size":8},
        {"name":"lower_lr","latent_dim":LATENT_DIM,"lr":LEARNING_RATE/2,"smooth":True},
        {"name":"no_label_smoothing","latent_dim":LATENT_DIM,"lr":LEARNING_RATE,"smooth":False},
    ]
    rows=[]
    old_latent,old_lr,old_batch=LATENT_DIM,LEARNING_RATE,BATCH_SIZE
    for cfg in configs:
        print("\\nRunning experiment:",cfg["name"])
        LATENT_DIM,LEARNING_RATE=cfg["latent_dim"],cfg["lr"]
        BATCH_SIZE=cfg.get("batch_size",old_batch)
        train_loader=DataLoader(train_subset,batch_size=BATCH_SIZE,shuffle=True,
                                num_workers=NUM_WORKERS,pin_memory=(device.type=="cuda"),
                                drop_last=False)
        val_loader=DataLoader(val_subset,batch_size=BATCH_SIZE,shuffle=False,
                              num_workers=NUM_WORKERS,pin_memory=(device.type=="cuda"),drop_last=False)
        test_loader=DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,
                               num_workers=NUM_WORKERS,pin_memory=(device.type=="cuda"),drop_last=False)
        fixed_noise=torch.randn(64,LATENT_DIM,device=device)
        G=VanillaGenerator(latent_dim=LATENT_DIM).to(device)
        D=VanillaDiscriminator().to(device)
        try:
            hist,_=train_gan("experiment_"+cfg["name"],G,D,epochs=epochs,label_smoothing=cfg["smooth"],learning_rate=LEARNING_RATE)
            rows.append({"experiment":cfg["name"],"latent_dim":LATENT_DIM,"batch_size":BATCH_SIZE,
                         "learning_rate":LEARNING_RATE,"label_smoothing":cfg["smooth"],"final_train_d_loss":hist["d_loss"][-1],
                         "final_train_g_loss":hist["g_loss"][-1],"final_val_d_loss":hist["val_d_loss"][-1],
                         "final_val_g_loss":hist["val_g_loss"][-1]})
        finally:
            del G,D
            if torch.cuda.is_available(): torch.cuda.empty_cache()
    LATENT_DIM,LEARNING_RATE,BATCH_SIZE=old_latent,old_lr,old_batch
    train_loader=DataLoader(train_subset,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,
                            pin_memory=(device.type=="cuda"),drop_last=False)
    val_loader=DataLoader(val_subset,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,
                          pin_memory=(device.type=="cuda"),drop_last=False)
    test_loader=DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,
                           pin_memory=(device.type=="cuda"),drop_last=False)
    fixed_noise=torch.randn(64,LATENT_DIM,device=device)
    results=pd.DataFrame(rows)
    results.to_csv(OUTPUT_DIR/"sensitivity_experiments.csv",index=False)
    display(results)
    return results

if RUN_SENSITIVITY_EXPERIMENTS:
    sensitivity_results=run_sensitivity_experiments(epochs=EPOCHS)
else:
    print("Sensitivity runs disabled for smoke test. Set RUN_SENSITIVITY_EXPERIMENTS=True to execute them.")


## 15. Compare models and save the report summary

The table contains measured run results. GAN losses alone do not establish sample quality; inspect grids and quantitative metrics too.


In [ ]:
comparison_rows = [
    {"Model":"Vanilla GAN","Final train D loss":vanilla_history["d_loss"][-1],
     "Final train G loss":vanilla_history["g_loss"][-1],**vanilla_metrics},
    {"Model":"DCGAN","Final train D loss":dcgan_history["d_loss"][-1],
     "Final train G loss":dcgan_history["g_loss"][-1],**dcgan_metrics},
]
if RUN_CONDITIONAL_DCGAN and "conditional_history" in globals():
    comparison_rows.append({
        "Model":"Conditional DCGAN",
        "Final train D loss":conditional_history["d_loss"][-1],
        "Final train G loss":conditional_history["g_loss"][-1],
        "metric_note":"Class-wise grids saved; conditional FID not calculated in this smoke-test template."
    })
comparison=pd.DataFrame(comparison_rows)
display(comparison)
comparison.to_csv(OUTPUT_DIR/"gan_comparison.csv",index=False)
with open(OUTPUT_DIR/"assignment2_summary.json","w") as f:
    json.dump({
        "image_size":IMAGE_SIZE,"latent_dim":LATENT_DIM,"batch_size":BATCH_SIZE,
        "epochs":EPOCHS,"learning_rate":LEARNING_RATE,"betas":[BETA1,BETA2],
        "seed":SEED,"class_names":class_names,
        "vanilla_metrics":vanilla_metrics,"dcgan_metrics":dcgan_metrics,
        "conditional_dcgan_trained":bool(RUN_CONDITIONAL_DCGAN and "conditional_history" in globals()),
        "notes":[
            "One epoch is a smoke test, not a final quality result.",
            "Validation generator loss is a weak checkpoint-selection proxy.",
            "If FID fails, report fallback as proxy, not FID.",
            "Metrics must be generated by running the notebook on the dataset."
        ]
    },f,indent=2)
print("Saved results to",OUTPUT_DIR)
